# Gold Layer — Fraud Analytics & Loss Prevention

Builds 4 Gold deliverables from `workspace.silver_schema.suspect_fraud_checks`:
1. `gold_employee_fraud_scorecard` — per-employee fraud flag aggregation
2. `gold_location_loss_prevention_summary` — per-location loss metrics
3. `gold_hourly_anomaly_trends` — hourly anomaly patterns by location
4. `gold_suspect_check_details` — audit view for receipt-level investigation

In [0]:
# ============================================================================
# Gold Layer — Imports, Configuration & Schema Setup
# ============================================================================
from pyspark.sql.functions import (
    col, count as _count, sum as _sum, avg as _avg,
    explode, when, lit, coalesce
)

SILVER_SCHEMA = "workspace.silver_schema"
GOLD_SCHEMA   = "workspace.gold_schema"

# Create the gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"✅ Schema '{GOLD_SCHEMA}' is ready.")

# Quick revenue_centers schema check for Table 2 join
print("\nrevenue_centers columns:")
for f in spark.table(f"{SILVER_SCHEMA}.revenue_centers").schema.fields:
    print(f"  {f.name}")
display(spark.table(f"{SILVER_SCHEMA}.revenue_centers").limit(5))

In [0]:
# ============================================================================
# TABLE 1: gold_employee_fraud_scorecard
# ============================================================================
# Groups suspect_fraud_checks by employee to build a per-employee fraud scorecard.
# EmployeeId is nested inside the Discounts array — we explode it to attribute
# each suspect check to the employee who applied a discount, then join with the
# employees dimension table for the human-readable name.
# ============================================================================

source = spark.table(f"{SILVER_SCHEMA}.suspect_fraud_checks")

# Explode the Discounts array to get the EmployeeId who applied each discount
exploded = (source
    .select(
        col("LocationId"),
        col("Id"),
        col("total_discount_amount"),
        col("total_void_amount"),
        col("is_fraud_high_value_excess_discount"),
        col("is_fraud_low_value_excess_discount"),
        col("is_fraud_irregular_hours"),
        col("is_fraud_negative_values"),
        col("is_fraud_suspicious_in_store"),
        explode(col("Discounts")).alias("d"),
    )
    .filter(col("d.EmployeeId").isNotNull())
    .select(
        col("d.EmployeeId").alias("employee_id"),
        col("LocationId"),
        col("Id"),
        col("total_discount_amount"),
        col("total_void_amount"),
        col("is_fraud_high_value_excess_discount"),
        col("is_fraud_low_value_excess_discount"),
        col("is_fraud_irregular_hours"),
        col("is_fraud_negative_values"),
        col("is_fraud_suspicious_in_store"),
    )
)

# Deduplicate to one row per (employee_id, LocationId, Id) so an employee who
# applied multiple discounts on the same check is not double-counted.
deduped = exploded.dropDuplicates(["employee_id", "LocationId", "Id"])

# Join with employees dimension for the employee name
emp_names = spark.table(f"{SILVER_SCHEMA}.employees").select(
    col("Id").alias("emp_id"),
    col("Name").alias("employee_name"),
)

# Aggregate fraud flags and financial risk per employee + location
scorecard = (deduped
    .join(emp_names, deduped.employee_id == emp_names.emp_id, "left")
    .drop("emp_id")
    .groupBy("employee_id", "employee_name", "LocationId")
    .agg(
        # Count of distinct suspect checks attributed to this employee
        _count("Id").alias("total_suspect_checks"),
        # Sum each fraud flag (cast boolean → 0/1) for per-hypothesis counts
        _sum(col("is_fraud_high_value_excess_discount").cast("int")).alias("high_val_discount_flags"),
        _sum(col("is_fraud_low_value_excess_discount").cast("int")).alias("low_val_discount_flags"),
        _sum(col("is_fraud_irregular_hours").cast("int")).alias("irregular_hours_flags"),
        _sum(col("is_fraud_negative_values").cast("int")).alias("negative_price_flags"),
        _sum(col("is_fraud_suspicious_in_store").cast("int")).alias("suspicious_togo_flags"),
        # Financial exposure: total discount and void dollars on suspect checks
        _sum("total_discount_amount").alias("total_discount_dollars_at_risk"),
        _sum("total_void_amount").alias("total_void_dollars_at_risk"),
    )
    .orderBy(col("total_suspect_checks").desc())
)

# Write as a Delta table with Liquid Clustering on LocationId for fast location-filtered queries
scorecard.createOrReplaceTempView("vw_employee_scorecard")
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.gold_employee_fraud_scorecard
    CLUSTER BY (LocationId)
    AS SELECT * FROM vw_employee_scorecard
""")

t1_count = spark.table(f"{GOLD_SCHEMA}.gold_employee_fraud_scorecard").count()
print(f"✅ {GOLD_SCHEMA}.gold_employee_fraud_scorecard — {t1_count:,} employee-location rows")

In [0]:
# ============================================================================
# TABLE 2: gold_location_loss_prevention_summary
# ============================================================================
# Groups suspect_fraud_checks by location and revenue center to quantify
# financial loss exposure.  Joins with the revenue_centers dimension table to
# translate RevenueCenterId into a human-readable name (e.g. "In-Store").
# ============================================================================

source = spark.table(f"{SILVER_SCHEMA}.suspect_fraud_checks")

# Join with revenue_centers to get the revenue center name
rc_names = spark.table(f"{SILVER_SCHEMA}.revenue_centers").select(
    col("Id").alias("rc_id"),
    col("Name").alias("revenue_center_name"),
)

# Aggregate loss metrics per location + revenue center
location_summary = (source
    .join(rc_names, source.RevenueCenterId == rc_names.rc_id, "left")
    .drop("rc_id")
    .groupBy("LocationId", "revenue_center_name")
    .agg(
        # Total suspect checks at this location + revenue center
        _count("Id").alias("total_flagged_checks"),
        # Total discount dollars at risk on suspect checks
        _sum("total_discount_amount").alias("total_discount_loss"),
        # Total void dollars at risk on suspect checks
        _sum("total_void_amount").alias("total_void_loss"),
        # Combined financial exposure: discounts + voids
        (_sum("total_discount_amount") + _sum("total_void_amount")).alias("total_dollars_at_risk"),
    )
    .orderBy(col("total_dollars_at_risk").desc())
)

# Write as Delta table with Liquid Clustering on LocationId
location_summary.createOrReplaceTempView("vw_location_summary")
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.gold_location_loss_prevention_summary
    CLUSTER BY (LocationId)
    AS SELECT * FROM vw_location_summary
""")

t2_count = spark.table(f"{GOLD_SCHEMA}.gold_location_loss_prevention_summary").count()
print(f"✅ {GOLD_SCHEMA}.gold_location_loss_prevention_summary — {t2_count:,} location-RC rows")

In [0]:
# ============================================================================
# TABLE 3: gold_hourly_anomaly_trends
# ============================================================================
# Groups suspect_fraud_checks by location and hour-of-day to reveal temporal
# fraud patterns.  Uses the `open_hour` column already derived in the silver
# layer (extracted from OpenTime via HOUR()).
# ============================================================================

source = spark.table(f"{SILVER_SCHEMA}.suspect_fraud_checks")

# Aggregate anomaly metrics per location + hour of day
hourly_trends = (source
    .groupBy("LocationId", col("open_hour").alias("hour_of_day"))
    .agg(
        # Total suspect checks opened in this hour
        _count("Id").alias("suspect_check_count"),
        # How many of those triggered the irregular-hours flag (2-7 AM)
        _sum(col("is_fraud_irregular_hours").cast("int")).alias("off_hours_count"),
        # How many triggered the suspicious in-store / to-go flag
        _sum(col("is_fraud_suspicious_in_store").cast("int")).alias("suspicious_togo_count"),
        # Average check duration for context
        _avg("check_duration_minutes").alias("avg_check_duration_minutes"),
    )
    .orderBy(col("LocationId").asc(), col("hour_of_day").asc())
)

# Write as Delta table with Liquid Clustering on LocationId
hourly_trends.createOrReplaceTempView("vw_hourly_trends")
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.gold_hourly_anomaly_trends
    CLUSTER BY (LocationId)
    AS SELECT * FROM vw_hourly_trends
""")

t3_count = spark.table(f"{GOLD_SCHEMA}.gold_hourly_anomaly_trends").count()
print(f"✅ {GOLD_SCHEMA}.gold_hourly_anomaly_trends — {t3_count:,} location-hour rows")

In [0]:
# ============================================================================
# VIEW 4: gold_suspect_check_details
# ============================================================================
# Creates a Delta View (CREATE OR REPLACE VIEW) that exposes all columns from
# the silver suspect_fraud_checks table under the gold schema.  This provides
# receipt-level audit access for detailed investigation without copying data.
# ============================================================================

spark.sql(f"""
    CREATE OR REPLACE VIEW {GOLD_SCHEMA}.gold_suspect_check_details
    AS SELECT * FROM {SILVER_SCHEMA}.suspect_fraud_checks
""")

view_count = spark.table(f"{GOLD_SCHEMA}.gold_suspect_check_details").count()
print(f"✅ {GOLD_SCHEMA}.gold_suspect_check_details — {view_count:,} rows (view over silver)")

In [0]:
# ============================================================================
# GOLD LAYER VERIFICATION
# ============================================================================
# Queries live tables to verify all 4 deliverables were created successfully.
# ============================================================================

print("=" * 72)
print("  Gold layer deliverables — workspace.gold_schema")
print("=" * 72)

gold_assets = [
    ("gold_employee_fraud_scorecard",     "TABLE"),
    ("gold_location_loss_prevention_summary", "TABLE"),
    ("gold_hourly_anomaly_trends",       "TABLE"),
    ("gold_suspect_check_details",        "VIEW"),
]

for name, asset_type in gold_assets:
    cnt = spark.table(f"{GOLD_SCHEMA}.{name}").count()
    print(f"  {name:<45} {cnt:>10,}  {asset_type}")
print("=" * 72)

# Preview Table 1: top 10 employees by suspect check count
print("\nTop 10 employees by suspect checks:")
display(spark.table(f"{GOLD_SCHEMA}.gold_employee_fraud_scorecard")
        .select("employee_id", "employee_name", "LocationId",
                "total_suspect_checks", "high_val_discount_flags",
                "low_val_discount_flags", "negative_price_flags",
                "total_discount_dollars_at_risk")
        .orderBy(col("total_suspect_checks").desc())
        .limit(10))

In [0]:
# ============================================================================
# PREVIEW: All 3 Gold tables (10 rows each)
# ============================================================================

print("── gold_employee_fraud_scorecard (top 10 by suspect checks) ──")
display(spark.table(f"{GOLD_SCHEMA}.gold_employee_fraud_scorecard")
        .orderBy(col("total_suspect_checks").desc())
        .limit(10))

print("\n── gold_location_loss_prevention_summary (top 10 by $ at risk) ──")
display(spark.table(f"{GOLD_SCHEMA}.gold_location_loss_prevention_summary")
        .orderBy(col("total_dollars_at_risk").desc())
        .limit(10))

print("\n── gold_hourly_anomaly_trends (first 10 by location + hour) ──")
display(spark.table(f"{GOLD_SCHEMA}.gold_hourly_anomaly_trends")
        .orderBy(col("LocationId").asc(), col("hour_of_day").asc())
        .limit(10))